# Data Collection - Web Scraping

BeautifulSoup extracts launch-table cells from a saved Wikipedia revision. Table and cell counts are retained for audit. The scraped source is analyzed separately from the course modeling checkpoint.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


def scrape():
    if not (DATA/'wikipedia_launches.html').exists():
        import urllib.request
        url='https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches_%282010%E2%80%932019%29'
        request=urllib.request.Request(url,headers={'User-Agent':'EducationalCapstone/1.0'})
        (DATA/'wikipedia_launches.html').write_bytes(urllib.request.urlopen(request,timeout=60).read())
    soup=BeautifulSoup((DATA/'wikipedia_launches.html').read_text(encoding='utf-8'),'html.parser')
    for ref in soup.select('sup.reference'): ref.decompose()
    rows=[]
    for ti,table in enumerate(soup.select('table.wikitable')):
        headers=[x.get_text(' ',strip=True) for x in table.select('tr')[:1] for x in x.select('th')]
        if not any('Flight' in h or 'Date' in h for h in headers): continue
        for tr in table.select('tr')[1:]:
            cells=tr.find_all(['td','th'],recursive=False)
            if len(cells)>=10:
                vals=[c.get_text(' ',strip=True) for c in cells]
                rows.append({'Table':ti,'Flight':vals[0],'Date':vals[1],
                    'Booster':vals[2],'LaunchSite':vals[3],'Payload':vals[4],
                    'PayloadMass':vals[5],'Orbit':vals[6],
                    'Customer':vals[7] if len(vals)>7 else '',
                    'LaunchOutcome':vals[8] if len(vals)>8 else '',
                    'LandingOutcome':vals[9] if len(vals)>9 else ''})
    out=pd.DataFrame(rows)
    assert len(out)>70, 'Launch-table extraction failed'
    out.to_csv(DATA/'web_scraped.csv',index=False)
    print('Wikipedia snapshot: extracted',len(out),'launch rows using BeautifulSoup')
    print(out.head().to_string(index=False)); return out


In [2]:
scraped = scrape()

Wikipedia snapshot: extracted 81 launch rows using BeautifulSoup
 Table Flight                  Date       Booster                LaunchSite                                 Payload                              PayloadMass       Orbit      Customer LaunchOutcome      LandingOutcome
     0      1     4 June 2010 18:45 F9 v1.0 B0003 Cape Canaveral , SLC ‑ 40    Dragon Spacecraft Qualification Unit                                  Unknown         LEO        SpaceX       Success Failure (parachute)
     0      2 8 December 2010 15:43 F9 v1.0 B0004 Cape Canaveral , SLC ‑ 40 SpaceX COTS Demo Flight 1 (Dragon C101)                                  Unknown         LEO NASA ( COTS )       Success Failure (parachute)
     0      3     22 May 2012 07:44 F9 v1.0 B0005 Cape Canaveral , SLC ‑ 40 SpaceX COTS Demo Flight 2 (Dragon C102)    525 kg (1,157 lb) (excl. Dragon mass) LEO ( ISS ) NASA ( COTS )       Success          No attempt
     0      4  8 October 2012 00:35 F9 v1.0 B0006 Cape Canaveral , 